# Engenharia de Features

Este notebook cria uma linha de features para cada partida do Brasileirão
entre 2020 e 2024. Execute antes `python src/data/prepare_matches.py` na raiz
do projeto e depois rode todas as células deste notebook em ordem.

O histórico é ordenado por temporada, data e ID antes de calcular os últimos
cinco jogos. Cada janela usa apenas resultados de partidas anteriores na
mesma temporada. Na primeira partida de cada clube (e no primeiro jogo em
cada mando), as médias ficam vazias porque ainda não há histórico. Esses
valores são mantidos como ausentes na base gerada.

In [ ]:
from pathlib import Path
import pandas as pd

caminho_partidas = Path('../data/processed/matches_2020_2024.csv')
if not caminho_partidas.is_file():
    raise FileNotFoundError(
        'Base processada ausente. Na raiz do projeto, execute: '
        'python src/data/prepare_matches.py'
    )

df = pd.read_csv(caminho_partidas, parse_dates=['data'])

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["temporada"].value_counts().sort_index()

In [ ]:
jogos_mandante = df[
    [
        "id",
        "temporada",
        "rodada",
        "data",
        "mandante",
        "visitante",
        "gols_mandante",
        "gols_visitante",
        "resultado"
    ]
].copy()

In [ ]:
jogos_mandante = jogos_mandante.rename(
    columns={
        "mandante": "clube",
        "visitante": "adversario",
        "gols_mandante": "gols_pro",
        "gols_visitante": "gols_contra"
    }
)

In [ ]:
jogos_mandante["mando"] = "casa"

In [ ]:
jogos_mandante["pontos"] = 0

jogos_mandante.loc[
    jogos_mandante["resultado"] == "H",
    "pontos"
] = 3

jogos_mandante.loc[
    jogos_mandante["resultado"] == "D",
    "pontos"
] = 1

In [ ]:
jogos_visitante = df[
    [
        "id",
        "temporada",
        "rodada",
        "data",
        "visitante",
        "mandante",
        "gols_visitante",
        "gols_mandante",
        "resultado"
    ]
].copy()

In [ ]:
jogos_visitante = jogos_visitante.rename(
    columns={
        "visitante": "clube",
        "mandante": "adversario",
        "gols_visitante": "gols_pro",
        "gols_mandante": "gols_contra"
    }
)

In [ ]:
jogos_visitante["mando"] = "fora"

In [ ]:
jogos_visitante["pontos"] = 0

jogos_visitante.loc[
    jogos_visitante["resultado"] == "A",
    "pontos"
] = 3

jogos_visitante.loc[
    jogos_visitante["resultado"] == "D",
    "pontos"
] = 1

In [ ]:
historico_clubes = (
    pd.concat([jogos_mandante, jogos_visitante], ignore_index=True)
    .sort_values(['temporada', 'data', 'id', 'mando'])
    .reset_index(drop=True)
)

In [ ]:
historico_clubes.shape

In [ ]:
historico_clubes["mando"].value_counts()

In [ ]:
historico_clubes[
    [
        "data",
        "clube",
        "adversario",
        "mando",
        "gols_pro",
        "gols_contra",
        "pontos"
    ]
].head(10)

In [ ]:
historico_clubes["pontos_jogo_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos"]
    .shift(1)
)

In [ ]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "pontos_jogo_anterior"
    ]
].head(10)

In [ ]:
historico_clubes["pontos_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos_jogo_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "pontos_jogo_anterior",
        "pontos_ultimos_5"
    ]
].head(10)

In [ ]:
historico_clubes["gols_pro_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_pro"]
    .shift(1)
)

In [ ]:
historico_clubes["gols_contra_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_contra"]
    .shift(1)
)

In [ ]:
historico_clubes["gols_pro_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_pro_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes["gols_contra_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["gols_contra_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "data",
        "rodada",
        "adversario",
        "gols_pro",
        "gols_contra",
        "pontos_ultimos_5",
        "gols_pro_ultimos_5",
        "gols_contra_ultimos_5"
    ]
].head(10)

In [ ]:
historico_clubes["jogos_anteriores_5"] = (
    historico_clubes
    .groupby(["temporada", "clube"])["pontos_jogo_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).count()
    )
)

In [ ]:
historico_clubes["pontos_por_jogo_ultimos_5"] = (
    historico_clubes["pontos_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [ ]:
historico_clubes["media_gols_pro_ultimos_5"] = (
    historico_clubes["gols_pro_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [ ]:
historico_clubes["media_gols_contra_ultimos_5"] = (
    historico_clubes["gols_contra_ultimos_5"]
    / historico_clubes["jogos_anteriores_5"]
)

In [ ]:
historico_clubes[
    (historico_clubes["clube"] == "Flamengo") &
    (historico_clubes["temporada"] == 2024)
][
    [
        "rodada",
        "pontos",
        "jogos_anteriores_5",
        "pontos_ultimos_5",
        "pontos_por_jogo_ultimos_5",
        "media_gols_pro_ultimos_5",
        "media_gols_contra_ultimos_5"
    ]
].head(10)

In [ ]:
historico_clubes["saldo_gols_ultimos_5"] = (
    historico_clubes["gols_pro_ultimos_5"]
    - historico_clubes["gols_contra_ultimos_5"]
)

In [ ]:
historico_clubes["media_saldo_gols_ultimos_5"] = (
    historico_clubes["media_gols_pro_ultimos_5"]
    - historico_clubes["media_gols_contra_ultimos_5"]
)

In [ ]:
historico_clubes["pontos_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos"]
    .shift(1)
)

In [ ]:
historico_clubes["jogos_mesmo_mando_anteriores_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).count()
    )
)

In [ ]:
historico_clubes["pontos_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["pontos_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes["pontos_por_jogo_mesmo_mando_ultimos_5"] = (
    historico_clubes["pontos_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

In [ ]:
historico_clubes[
    (historico_clubes["clube"] == "Fortaleza") &
    (historico_clubes["temporada"] == 2024) &
    (historico_clubes["mando"] == "casa")
][
    [
        "data",
        "rodada",
        "adversario",
        "pontos",
        "jogos_mesmo_mando_anteriores_5",
        "pontos_mesmo_mando_ultimos_5",
        "pontos_por_jogo_mesmo_mando_ultimos_5"
    ]
].head(10)

In [ ]:
historico_clubes["gols_pro_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_pro"]
    .shift(1)
)

historico_clubes["gols_contra_mesmo_mando_anterior"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_contra"]
    .shift(1)
)

In [ ]:
historico_clubes["gols_pro_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_pro_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes["gols_contra_mesmo_mando_ultimos_5"] = (
    historico_clubes
    .groupby(["temporada", "clube", "mando"])["gols_contra_mesmo_mando_anterior"]
    .transform(
        lambda serie: serie.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

In [ ]:
historico_clubes["media_gols_pro_mesmo_mando_ultimos_5"] = (
    historico_clubes["gols_pro_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

historico_clubes["media_gols_contra_mesmo_mando_ultimos_5"] = (
    historico_clubes["gols_contra_mesmo_mando_ultimos_5"]
    / historico_clubes["jogos_mesmo_mando_anteriores_5"]
)

In [ ]:
historico_clubes["media_saldo_mesmo_mando_ultimos_5"] = (
    historico_clubes["media_gols_pro_mesmo_mando_ultimos_5"]
    - historico_clubes["media_gols_contra_mesmo_mando_ultimos_5"]
)

In [ ]:
fortaleza_casa = (
    historico_clubes[
        (historico_clubes["clube"] == "Fortaleza") &
        (historico_clubes["temporada"] == 2024) &
        (historico_clubes["mando"] == "casa")
    ]
    .sort_values(["data", "id"])
    .reset_index(drop=True)
)

# Vamos testar o 6º jogo em casa.
# Portanto, já existem exatamente 5 jogos anteriores.
i = 5

jogo_atual = fortaleza_casa.iloc[i]

ultimos_5 = fortaleza_casa.iloc[i-5:i]

print("PARTIDA ATUAL")
print(
    jogo_atual[
        ["data", "rodada", "adversario", "gols_pro", "gols_contra", "pontos"]
    ]
)

print("\n5 JOGOS ANTERIORES")
print(
    ultimos_5[
        ["data", "adversario", "gols_pro", "gols_contra", "pontos"]
    ]
)

# Cálculo manual usando somente os jogos anteriores
pontos_manual = ultimos_5["pontos"].sum()
gols_pro_manual = ultimos_5["gols_pro"].sum()
gols_contra_manual = ultimos_5["gols_contra"].sum()

media_pontos_manual = pontos_manual / len(ultimos_5)
media_gols_pro_manual = gols_pro_manual / len(ultimos_5)
media_gols_contra_manual = gols_contra_manual / len(ultimos_5)

print("\nCOMPARAÇÃO")

print(
    "Pontos:",
    pontos_manual,
    "==",
    jogo_atual["pontos_mesmo_mando_ultimos_5"]
)

print(
    "Média pontos:",
    media_pontos_manual,
    "==",
    jogo_atual["pontos_por_jogo_mesmo_mando_ultimos_5"]
)

print(
    "Gols pró:",
    gols_pro_manual,
    "==",
    jogo_atual["gols_pro_mesmo_mando_ultimos_5"]
)

print(
    "Média gols pró:",
    media_gols_pro_manual,
    "==",
    jogo_atual["media_gols_pro_mesmo_mando_ultimos_5"]
)

print(
    "Gols contra:",
    gols_contra_manual,
    "==",
    jogo_atual["gols_contra_mesmo_mando_ultimos_5"]
)

print(
    "Média gols contra:",
    media_gols_contra_manual,
    "==",
    jogo_atual["media_gols_contra_mesmo_mando_ultimos_5"]
)

In [ ]:
assert pontos_manual == jogo_atual["pontos_mesmo_mando_ultimos_5"]

assert abs(
    media_pontos_manual
    - jogo_atual["pontos_por_jogo_mesmo_mando_ultimos_5"]
) < 1e-9

assert gols_pro_manual == jogo_atual["gols_pro_mesmo_mando_ultimos_5"]

assert abs(
    media_gols_pro_manual
    - jogo_atual["media_gols_pro_mesmo_mando_ultimos_5"]
) < 1e-9

assert gols_contra_manual == jogo_atual["gols_contra_mesmo_mando_ultimos_5"]

assert abs(
    media_gols_contra_manual
    - jogo_atual["media_gols_contra_mesmo_mando_ultimos_5"]
) < 1e-9

print("Todas as features conferem.")

## Conferência da ordem cronológica

O exemplo acima confere jogos no mesmo mando. A checagem abaixo confere os
cinco jogos anteriores do clube considerando casa e fora, para detectar
janelas que teriam sido calculadas com os dois blocos fora de ordem.

In [ ]:
flamengo_2024 = historico_clubes[
    (historico_clubes['clube'] == 'Flamengo')
    & (historico_clubes['temporada'] == 2024)
].reset_index(drop=True)
jogo_atual_geral = flamengo_2024.iloc[5]
cinco_anteriores = flamengo_2024.iloc[:5]

assert (cinco_anteriores['data'] < jogo_atual_geral['data']).all()
assert cinco_anteriores['pontos'].sum() == jogo_atual_geral['pontos_ultimos_5']
assert cinco_anteriores['gols_pro'].sum() == jogo_atual_geral['gols_pro_ultimos_5']
assert cinco_anteriores['gols_contra'].sum() == jogo_atual_geral['gols_contra_ultimos_5']
print('Histórico geral em ordem cronológica conferido.')

In [ ]:
features_base = [
    "pontos_por_jogo_ultimos_5",
    "media_gols_pro_ultimos_5",
    "media_gols_contra_ultimos_5",
    "pontos_por_jogo_mesmo_mando_ultimos_5",
    "media_gols_pro_mesmo_mando_ultimos_5",
    "media_gols_contra_mesmo_mando_ultimos_5",
]

In [ ]:
features_mandante = historico_clubes[
    historico_clubes["mando"] == "casa"
][
    ["id"] + features_base
].copy()

In [ ]:
features_mandante = features_mandante.rename(
    columns={
        feature: f"mandante_{feature}"
        for feature in features_base
    }
)

## Uma linha por partida

Selecionamos as mesmas seis métricas para mandante e visitante e as
juntamos ao jogo pelo ID. `resultado` é o alvo futuro; gols da própria
partida não entram nas features. A junção exige um registro de cada
lado para cada jogo.

In [ ]:
features_visitante = historico_clubes[
    historico_clubes['mando'] == 'fora'
][['id'] + features_base].copy()
features_visitante = features_visitante.rename(
    columns={feature: f'visitante_{feature}' for feature in features_base}
)

In [ ]:
base_features = (
    df[['id', 'temporada', 'rodada', 'data', 'mandante', 'visitante', 'resultado']]
    .merge(features_mandante, on='id', validate='one_to_one')
    .merge(features_visitante, on='id', validate='one_to_one')
    .sort_values(['temporada', 'data', 'id'])
    .reset_index(drop=True)
)
assert len(base_features) == len(df) == 1900
assert base_features['id'].is_unique
base_features.shape

In [ ]:
base_features.head(10)

In [ ]:
colunas_features = [
    f'{lado}_{feature}'
    for lado in ('mandante', 'visitante')
    for feature in features_base
]
base_features[colunas_features].isna().sum()

In [ ]:
caminho_features = Path('../data/processed/matches_features_2020_2024.csv')
base_features.to_csv(caminho_features, index=False)
print(f'Base de features salva em: {caminho_features}')
print(f'Partidas: {len(base_features)} | Features: {len(colunas_features)}')